# 12. Molecular Images with RDKit -- SMILES to Pictures

SMILES is a text representation of molecules, but chemists think in structures. In this notebook we learn to convert SMILES strings into molecular images programmatically using **RDKit**, the most widely used open-source cheminformatics toolkit.

Along the way, we will see that images are just NumPy arrays -- connecting directly to what we learned in Notebook 11.

**Topics covered:**
- RDKit `Mol` objects from SMILES
- Drawing 2D molecular structures
- Images as NumPy arrays (grayscale conversion, pixel histograms)
- Batch processing: saving images and building image arrays
- (Optional) RDKit molecular descriptors

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import os

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, AllChem

## 12.1 Loading Our Polymer Data

In [ ]:
from pathlib import Path

ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ffv_clean_path = ARTIFACT_DIR / "ffv_clean.csv"

if not ffv_clean_path.exists():
    print("Rebuilding artifacts/ffv_clean.csv from data/train.csv ...")
    train_df = pd.read_csv("data/train.csv")
    ffv_df = train_df.dropna(subset=["FFV"])[["id", "SMILES", "FFV"]].copy()
    ffv_df.to_csv(ffv_clean_path, index=False)
    print(f"Saved {ffv_clean_path} with {len(ffv_df)} rows.")
else:
    print(f"Found existing {ffv_clean_path} -- skipping rebuild.")

In [ ]:
df = pd.read_csv("artifacts/ffv_clean.csv")
print(f"Dataset: {len(df)} polymers")
df.head()

Let's pick one SMILES string to work with:

In [ ]:
example_smiles = df["SMILES"].iloc[0]
print("SMILES:", example_smiles)

Can you draw this molecule by looking at the SMILES string? Let's let RDKit do it for us.

---
## 12.2 RDKit Basics: Mol Objects

The central object in RDKit is a `Mol` -- it represents the molecular graph (atoms and bonds). We create one from a SMILES string:

In [ ]:
mol = Chem.MolFromSmiles(example_smiles)
print("Type:", type(mol))
print("Number of atoms:", mol.GetNumAtoms())
print("Number of bonds:", mol.GetNumBonds())

### Handling invalid SMILES

If a SMILES string is invalid, `MolFromSmiles` returns `None`. Always check!

In [ ]:
bad_mol = Chem.MolFromSmiles("this_is_not_a_smiles")
print("Result for invalid SMILES:", bad_mol)

### Exercise 1

1. Pick 5 SMILES strings from the dataframe (e.g. rows 0, 10, 50, 100, 500).
2. Convert each to a `Mol` object.
3. Print the SMILES and the number of atoms for each.
4. Check if any return `None`.

In [ ]:
# Your code here


---
## 12.3 Drawing Molecules

RDKit can generate a 2D depiction of any molecule:

In [ ]:
img = Draw.MolToImage(mol, size=(400, 400))
img

`MolToImage` returns a **PIL Image** object. We can convert it to a NumPy array to inspect it:

In [ ]:
img_array = np.array(img)
print("Type:", type(img_array))
print("Shape:", img_array.shape)   # (height, width, channels)
print("Dtype:", img_array.dtype)

The shape is `(400, 400, 3)` -- this is the 3D NumPy array from Notebook 11! Height x Width x RGB channels.

What colour is the pixel at the center?

In [ ]:
# RGB value of center pixel
print("Center pixel RGB:", img_array[200, 200, :])

### Drawing a grid of molecules

We can draw multiple molecules at once with `MolsToGridImage`:

In [ ]:
# Draw 8 random polymers
sample = df.sample(8, random_state=42)
mols = [Chem.MolFromSmiles(s) for s in sample["SMILES"]]
legends = [f"FFV={ffv:.3f}" for ffv in sample["FFV"]]

grid_img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(300, 300), legends=legends)
grid_img

### Exercise 2

1. Find the 4 polymers with the **highest** FFV and the 4 with the **lowest** FFV.
2. Draw them in a grid (8 molecules, 4 per row).
3. Include the FFV value as a legend under each molecule.
4. Can you visually spot any structural differences between high-FFV and low-FFV polymers?

In [ ]:
# Your code here


---
## 12.4 Images as NumPy Arrays

Since an image is just a NumPy array, we can use everything from Notebook 11 to manipulate it.

### Converting to grayscale

A simple way to convert RGB to grayscale is to average the 3 colour channels:

In [ ]:
gray = np.mean(img_array[:, :, :3], axis=2)  # axis=2 collapses the RGB channels
print("Grayscale shape:", gray.shape)  # (400, 400)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_array)
axes[0].set_title("RGB")
axes[1].imshow(gray, cmap="gray")
axes[1].set_title("Grayscale")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Splitting an image into its colour channels

The molecular image we drew is mostly white, so it is not the best example to look at the individual Red, Green, and Blue channels. Let's load a more colourful test image (a coffee photo from `scikit-image`) and split it into channels.

In [ ]:
from skimage import data

rgb = data.coffee()              # a colourful (400, 600, 3) RGB image
print("Shape:", rgb.shape, "dtype:", rgb.dtype)

# Each channel is a 2D slice along axis=2
R = rgb[:, :, 0]
G = rgb[:, :, 1]
B = rgb[:, :, 2]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(rgb);            axes[0].set_title("Original RGB")
axes[1].imshow(R, cmap="Reds");   axes[1].set_title("Red channel")
axes[2].imshow(G, cmap="Greens"); axes[2].set_title("Green channel")
axes[3].imshow(B, cmap="Blues");  axes[3].set_title("Blue channel")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# Brighter pixels in a channel = more of that colour at that location.
# E.g. the red coffee cup lights up the Red channel; the white tablecloth
# is bright in all three channels (white = R + G + B all high).

### Pixel intensity histogram

In [ ]:
plt.figure(figsize=(6, 3))
plt.hist(gray.ravel(), bins=50, color="steelblue", edgecolor="white")
plt.xlabel("Pixel intensity")
plt.ylabel("Count")
plt.title("Pixel intensity distribution")
# plt.yscale("log")  # Log scale for better visibility
plt.tight_layout()
plt.show()

Most pixels are white (intensity ~255, the background). The dark pixels are where the molecule is drawn.

### Thresholding -- finding molecule pixels

In [ ]:
# Create a binary mask: True where the molecule is drawn
threshold = 200
molecule_mask = gray < threshold

molecule_fraction = np.sum(molecule_mask) / molecule_mask.size
print(f"Fraction of image covered by molecule: {molecule_fraction:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Grayscale")
axes[1].imshow(molecule_mask, cmap="gray")
axes[1].set_title(f"Molecule mask (threshold={threshold})")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

### Exercise 3

1. Pick 10 polymers with different SMILES lengths (short to long).
2. For each, generate a 200x200 image, convert to grayscale, and compute the molecule fraction (fraction of pixels below threshold 200).
3. Plot molecule fraction vs. SMILES length. Do longer SMILES strings produce more complex images?

In [ ]:
# Your code here


---
## 12.5 Saving Images and Batch Processing

For machine learning, we often need to process many images at once. Let's build a pipeline.

### Saving a single image

In [ ]:
# Create output directory
os.makedirs("mol_images", exist_ok=True)

# Save the image
polymer_id = df["id"].iloc[0]
img.save(f"mol_images/polymer_{polymer_id}.png")
print(f"Saved: mol_images/polymer_{polymer_id}.png")

### Loading an image from file

Once an image is saved to disk you can load it back with PIL or matplotlib. Both give you a NumPy array you can process immediately.

In [ ]:
# --- Loading an image from file ---

# Option 1: PIL (gives a PIL Image object, same as MolToImage returns)
loaded_pil = Image.open(f"mol_images/polymer_{polymer_id}.png")
print("PIL image size:", loaded_pil.size)   # (width, height)

# Option 2: matplotlib (gives a NumPy array directly)
loaded_arr = plt.imread(f"mol_images/polymer_{polymer_id}.png")
print("NumPy array shape:", loaded_arr.shape, "dtype:", loaded_arr.dtype)

# Display it
plt.imshow(loaded_arr)
plt.axis("off")
plt.title("Loaded from disk")
plt.show()

# Both routes give you a NumPy array you can work with:
arr_from_pil = np.array(loaded_pil)
print("From PIL -> NumPy:", arr_from_pil.shape)

### Batch processing: SMILES to NumPy array

Let's write a function that converts a SMILES string into a fixed-size grayscale NumPy array:

In [ ]:
def smiles_to_grayscale(smiles, size=100):
    """Convert a SMILES string to a grayscale NumPy array."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.ones((size, size)) * 255  # white image for invalid SMILES
    img = Draw.MolToImage(mol, size=(size, size))
    img_array = np.array(img)[:, :, :3]  # drop alpha if present
    gray = np.mean(img_array, axis=2)
    return gray

In [ ]:
# Test it
test_img = smiles_to_grayscale(example_smiles)
print("Output shape:", test_img.shape)
plt.imshow(test_img, cmap="gray")
plt.axis("off")
plt.show()

### Building an image batch

Now let's convert 50 polymers into a single stacked array -- the format a neural network would expect:

In [ ]:
n_samples = 50
image_size = 100

sample_df = df.head(n_samples)
images = [smiles_to_grayscale(s, size=image_size) for s in sample_df["SMILES"]]
image_batch = np.stack(images)

print(f"Image batch shape: {image_batch.shape}")  # (50, 100, 100)
print(f"Memory: {image_batch.nbytes / 1024:.1f} KB")

In [ ]:
# Visualize a few from the batch
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(image_batch[i], cmap="gray")
    ax.set_title(f"FFV={sample_df['FFV'].iloc[i]:.3f}", fontsize=8)
    ax.axis("off")
plt.suptitle("First 10 polymer images from the batch")
plt.tight_layout()
plt.show()

### Exercise 4

1. Save 20 polymer images (200x200, RGB) to the `mol_images/` folder.
2. Name them `polymer_<id>.png` where `<id>` comes from the dataframe.
3. Print a confirmation for each saved file.

In [ ]:
# Your code here


---
## 12.6 (Optional) RDKit Molecular Descriptors

Beyond images, RDKit can compute **numerical descriptors** that summarize molecular properties. In CodingTask 1 some of you already used a few of these. Here is a quick tour:

In [ ]:
mol = Chem.MolFromSmiles(example_smiles)

print(f"Molecular Weight:     {Descriptors.MolWt(mol):.2f}")
print(f"LogP:                 {Descriptors.MolLogP(mol):.2f}")
print(f"TPSA:                 {Descriptors.TPSA(mol):.2f}")
print(f"Rotatable Bonds:      {Descriptors.NumRotatableBonds(mol)}")
print(f"H-Bond Donors:        {Descriptors.NumHDonors(mol)}")
print(f"H-Bond Acceptors:     {Descriptors.NumHAcceptors(mol)}")
print(f"Ring Count:           {Descriptors.RingCount(mol)}")

### Morgan fingerprints (a preview)

In professional cheminformatics, molecules are often encoded as **fingerprints** -- binary vectors that capture the presence of molecular substructures. This is how state-of-the-art ML on molecules works:

In [ ]:
fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024)
fp_array = np.array(fp)
print(f"Fingerprint shape: {fp_array.shape}")   # (1024,)
print(f"Non-zero bits:     {np.sum(fp_array)}")
print(f"First 50 bits:     {fp_array[:50]}")

Each bit encodes whether a particular molecular neighbourhood (radius=2 bonds) exists in the molecule. This 1024-dimensional vector can be used directly as input to an ML model. We won't use fingerprints in this course, but it's good to know they exist!

---
## 12.7 Summary

| Step | What we did |
|------|-------------|
| SMILES -> Mol | `Chem.MolFromSmiles(smiles)` |
| Mol -> Image | `Draw.MolToImage(mol, size=(w, h))` |
| Image -> NumPy | `np.array(img)` -- shape (H, W, 3) |
| RGB -> Grayscale | `np.mean(img_array[:,:,:3], axis=2)` |
| Batch of images | `np.stack(list_of_arrays)` -- shape (N, H, W) |
| Descriptors | `Descriptors.MolWt(mol)`, `.MolLogP()`, `.TPSA()`, etc. |

**Next up:** In Notebook 13, we will feed the numerical features into scikit-learn models to actually **predict** polymer properties!